**© Copyright AIDENTIFY. All rights reserved.**

# Part 4 | Session 05: Advanced RAG - HyDE, Reranking, Ensemble Retriever

## 📋 학습 목표

1️⃣ 기본 RAG의 한계점을 이해한다
2️⃣ **검색 품질을 정답 라벨로 측정하는 방법(Hit@k, MRR)을 익힌다**
3️⃣ HyDE (Hypothetical Document Embeddings)를 구현한다
4️⃣ Reranking으로 검색 결과 품질을 향상시킨다
5️⃣ Ensemble Retriever (BM25 + 시맨틱)를 구현한다
6️⃣ Parent Document Retriever를 이해하고 구현한다
7️⃣ **각 기법이 실제로 효과가 있는지 같은 벤치마크로 검증하고, 효과가 없다면 그 원인을 진단한다**

---

### ⚠️ 이 세션의 핵심 메시지

> **Advanced RAG 기법은 켠다고 좋아지지 않습니다.**
> 이 노트북에서 네 가지 기법을 측정합니다. 실측 결과 **하나는 확실히 이기고(+23%),
> 하나는 설정을 맞춰야만 이기며(+6%, 잘못 두면 -13%), 하나는 제자리(+2%),
> 하나는 실행할 때마다 결과가 흔들립니다.**
> 왜 그런지 원인까지 진단하는 것이 이 세션의 목표입니다.

기법을 켜기 전에 반드시 필요한 것이 둘 있습니다.

1. **한국어를 아는 모델** — 임베더도 리랭커도 영어 전용 모델을 쓰면 검색이 무너집니다
2. **정직한 평가 지표** — 검색기의 목적함수를 그대로 평가에 쓰면 베이스라인이 항상 이깁니다

---

### 🖥️ 실습 환경
- **GPU**: 선택사항 (Reranker는 GPU에서 훨씬 빠름, CPU도 가능)
- **필수 패키지**: langchain, chromadb, sentence-transformers, rank_bm25, langchain-openai
- **모델 다운로드**: 최초 실행 시 리랭커(약 2.2GB) 포함 약 2.7GB
- **HyDE 섹션만 OpenAI API 키 필요** (없으면 해당 셀만 건너뛰면 됩니다)


In [ ]:
# 📦 패키지 확인
import importlib
import os

packages = [
    "langchain",
    "langchain_community",
    "chromadb",
    "sentence_transformers",
    "rank_bm25",
    "langchain_openai",   # HyDE 섹션에서만 사용
]

print("📦 패키지 버전 확인")
print("=" * 40)
for pkg_name in packages:
    try:
        pkg = importlib.import_module(pkg_name)
        version = getattr(pkg, "__version__", "installed")
        print(f"  ✅ {pkg_name}: {version}")
    except ImportError:
        print(f"  ❌ {pkg_name}: 설치 필요")
        print(f"     pip install {pkg_name.replace('_', '-')}")

In [ ]:
# 🔧 GPU 메모리 유틸리티 (GPU 사용 시)
import torch, gc

def print_gpu_memory(tag=""):
    """GPU 메모리 사용량을 출력하는 유틸리티 함수"""
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / 1024**3
        total = torch.cuda.get_device_properties(0).total_memory / 1024**3
        print(f"[{tag}] GPU: {allocated:.1f}GB / {total:.1f}GB")
    else:
        print(f"[{tag}] CPU 모드로 실행 중")

# 리랭커는 GPU에서 훨씬 빠르다
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print_gpu_memory("초기 상태")
print(f"사용할 디바이스: {DEVICE}")

---

## 1️⃣ 기본 RAG의 한계

기본 RAG는 단순 유사도 검색에 의존하기 때문에 여러 한계가 있습니다.

### 🔴 기본 RAG의 문제점

| 문제 | 설명 | 해결 기법 |
|------|------|----------|
| 질문-문서 불일치 | 질문 형태와 문서 형태가 다름 | **HyDE** |
| 검색 품질 저하 | Top-k 결과 중 관련 없는 문서 포함 | **Reranking** |
| 키워드 누락 | 의미는 같지만 단어가 다른 경우 | **Ensemble Retriever** |
| 문맥 손실 | 작은 청크로 인한 맥락 부족 | **Parent Document Retriever** |

### 📊 Advanced RAG 기법 체계

```
Advanced RAG
├── Pre-Retrieval (검색 전)
│   ├── HyDE (가상 문서 생성)
│   └── Query Expansion (쿼리 확장)
├── Retrieval (검색)
│   ├── Ensemble Retriever (BM25 + 시맨틱)
│   └── Parent Document Retriever
└── Post-Retrieval (검색 후)
    ├── Reranking (재순위화)
    └── Context Compression (컨텍스트 압축)
```

> 위 표의 "해결 기법" 열은 **주장**입니다. 이 노트북의 나머지 전부는 그 주장을 검증하는 데 씁니다.
> 검증하려면 먼저 **무엇을 개선이라 부를지** 정해야 합니다. 그것이 2️⃣의 주제입니다.


---

## 2️⃣ 평가 방법론: 무엇을 "개선"이라 부를 것인가

Advanced RAG를 다루는 자료에서 가장 흔한 실수가 여기서 나옵니다.
**측정을 잘못하면 모든 기법이 베이스라인보다 나빠 보입니다.**

### 🚫 하면 안 되는 평가: 검색기의 목적함수를 평가에 그대로 쓰기

이런 코드를 자주 봅니다.

```python
# ❌ 이렇게 하면 안 된다
score = mean(cosine(query_emb, doc_emb) for doc in retrieved_docs)
```

문제는 **기본 retriever가 하는 일이 정확히 이 값을 최대화하는 것**이라는 데 있습니다.

```
기본 retriever : cosine(query, doc) 가 가장 큰 문서 3개를 고른다
평가 지표      : 고른 문서 3개의 mean cosine(query, doc)
                 ↑ 완전히 같은 식
```

즉 **기본 RAG는 정의상 이 지표의 최댓값**입니다. 청크가 10개면 3개를 뽑는 조합이 120가지인데,
기본 RAG는 그 중 최고점 조합을 매번 정확히 집어냅니다. 다른 어떤 기법도 이길 수 없고,
잘해야 비깁니다. **경기 전에 결과가 정해진 벤치마크**입니다.

여기에 임베딩 모델까지 검색기와 같은 것을 쓰면 순환 논리가 완성됩니다.

### ✅ 올바른 평가: 정답 라벨 + 순위 지표

검색 품질은 **"정답 문서를 위로 올렸는가"** 로 재야 합니다. 그러려면 질문마다 정답 출처가 필요합니다.

| 지표 | 정의 | 무엇을 보나 |
|------|------|-------------|
| **Hit@1** | 1위가 정답인 질문의 비율 | 가장 엄격. 최상위 정확도 |
| **Hit@3** | 정답이 top-3 안에 있는 비율 | LLM 컨텍스트에 근거가 들어갔는가 |
| **MRR@k** | 정답 순위의 역수 평균 (1위=1.0, 2위=0.5, 3위=0.33) | 순위까지 반영한 종합 점수 |

```
질문: "GRPO가 뭐야?"      정답 출처: rl_methods.txt

검색 결과 1위 rl_methods.txt  → Hit@1 ✓, 순위 1 → MRR 기여 1.00
검색 결과 2위 alignment.txt
검색 결과 3위 llm_basics.txt
```

**RAG에서는 Hit@3이 특히 중요합니다.** 검색된 문서는 전부 LLM 컨텍스트로 들어가므로,
1위가 아니어도 top-3 안에만 있으면 LLM이 답을 만들 재료는 확보한 셈입니다.

### 🔬 이 노트북의 벤치마크 설계

- **코퍼스 24개 문서** — 주제가 서로 겹치도록 구성 (LLM/파인튜닝/경량화/벡터DB/임베딩/프롬프트/강화학습/RAG/평가/에이전트)
- **질문 20개** — 정답 출처를 사람이 직접 라벨링
- 질문 유형을 일부러 섞었습니다:
  - **일반 질문** — "문서를 얼마나 잘게 쪼개야 하나?"
  - **고유명사 질문** — "GRPO가 뭐야?", "GraphQL을 쓰는 검색 엔진은?" → BM25가 유리할 수 있음
  - **어휘 불일치 질문** — "단계별로 생각하게 시키면 정답률이 오르나?" (문서에는 *Chain-of-Thought*로만 등장)

> 💡 문서를 5개만 쓰면 top-3이 코퍼스의 30%라, 기본 RAG의 Hit@3이 100%로 포화됩니다.
> 그러면 어떤 기법도 개선을 보일 수 없습니다. **개선의 여지가 있는 코퍼스**를 쓰는 것도 벤치마크 설계의 일부입니다.


In [ ]:
# 📄 벤치마크 코퍼스 + 정답 라벨
from langchain.schema import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter

# --- 코퍼스: 주제가 서로 겹치도록 24개 문서 구성 ---
# (source, topic, content) — 주제가 인접해야 "헷갈릴 여지"가 생기고, 그래야 기법 차이가 드러난다
RAW_DOCS = [
("llm_basics.txt", "LLM", """대규모 언어 모델(LLM)은 수십억 개의 파라미터를 가진 딥러닝 모델입니다.
GPT-4, Claude, Gemini 등이 대표적인 LLM이며, 대량의 텍스트 코퍼스로 사전학습됩니다.
사전학습 단계에서는 다음 토큰을 예측하는 단순한 목표로 언어의 통계적 구조를 습득합니다.
이후 지시 조정(instruction tuning)을 거쳐 사람의 지시를 따르는 형태로 다듬어집니다.
LLM의 핵심 능력은 긴 문맥을 이해하고 자연스러운 텍스트를 생성하는 것입니다.
파라미터 수가 커질수록 성능이 향상되는 경향을 스케일링 법칙이라 부릅니다.
다만 최근에는 무조건 크기를 키우기보다 데이터 품질을 높이는 방향이 중시됩니다."""),
("slm.txt", "LLM", """소형 언어 모델(sLLM)은 1B에서 7B 규모의 경량 언어 모델을 가리킵니다.
Phi-3, Gemma, Qwen2.5 등이 대표적이며 온디바이스 추론과 저비용 서빙에 적합합니다.
큰 모델 대비 성능 손실이 크지 않아 특정 도메인에서는 충분한 대안이 됩니다.
소형 모델은 응답 지연이 짧고 GPU 메모리를 적게 써서 동시 처리량이 높습니다.
개인정보를 외부로 보내지 않아도 되므로 규제가 엄격한 환경에서 선호됩니다.
반면 복잡한 다단계 추론이나 희귀 지식 질의에서는 대형 모델에 뒤처집니다.
따라서 작업 난이도에 따라 대형 모델과 소형 모델을 혼합해 쓰는 전략이 일반적입니다."""),
("transformer.txt", "아키텍처", """트랜스포머는 셀프 어텐션을 기반으로 한 신경망 아키텍처입니다.
2017년 인코더-디코더 구조로 제안되었으며 순환 신경망을 대체했습니다.
GPT 계열은 디코더만 사용하고 BERT 계열은 인코더만 사용합니다.
디코더 전용 구조는 이전 토큰만 참조하는 인과적 마스킹을 적용합니다.
포지셔널 인코딩으로 토큰의 순서 정보를 벡터에 주입합니다.
최근에는 회전 위치 임베딩(RoPE)이 긴 문맥 확장에 널리 쓰입니다.
잔차 연결과 층 정규화가 깊은 층을 안정적으로 학습시키는 역할을 합니다."""),
("attention.txt", "아키텍처", """셀프 어텐션은 Query, Key, Value 세 행렬의 곱으로 토큰 간 관계를 계산합니다.
Query와 Key의 내적으로 유사도를 구한 뒤 소프트맥스를 취해 가중치를 만듭니다.
멀티헤드 어텐션은 여러 부분공간에서 병렬로 어텐션을 수행해 표현력을 높입니다.
어텐션의 계산량은 시퀀스 길이의 제곱에 비례해 긴 입력에서 병목이 됩니다.
Flash Attention은 GPU 메모리 접근 패턴을 최적화해 긴 시퀀스 학습 속도를 크게 높입니다.
그룹 쿼리 어텐션(GQA)은 Key와 Value 헤드를 공유해 추론 메모리를 줄입니다.
슬라이딩 윈도우 어텐션은 참조 범위를 제한해 계산량을 선형에 가깝게 낮춥니다."""),
("finetuning.txt", "파인튜닝", """파인튜닝은 사전학습된 모델을 특정 작업에 맞게 추가 학습하는 방법입니다.
Full Fine-Tuning은 모든 파라미터를 업데이트하며 가장 높은 성능을 낼 수 있습니다.
대신 모델 크기만큼의 옵티마이저 상태가 필요해 리소스 요구가 매우 큽니다.
학습률, 배치 크기, 에폭 수 등 하이퍼파라미터 설정이 결과를 크게 좌우합니다.
데이터가 적으면 과적합이 쉽게 발생하므로 검증 손실을 함께 모니터링해야 합니다.
지시 데이터셋의 품질이 수량보다 중요하다는 것이 여러 연구의 공통된 결론입니다.
도메인 지식 주입이 목적이라면 파인튜닝보다 RAG가 더 저렴한 경우가 많습니다."""),
("peft.txt", "파인튜닝", """LoRA는 저랭크 행렬 분해로 학습 파라미터 수를 대폭 줄이는 PEFT 기법입니다.
원본 가중치는 얼려 두고 작은 어댑터 행렬 두 개만 학습해 저장 용량을 아낍니다.
QLoRA는 4bit 양자화와 LoRA를 결합해 메모리 요구를 한층 더 낮춥니다.
덕분에 RTX 4060 같은 소비자급 GPU에서도 7B 모델 학습이 가능해집니다.
랭크 r과 알파 값이 학습 용량을 결정하며 보통 8에서 64 사이를 사용합니다.
어댑터, 프리픽스 튜닝, IA3 등도 대표적인 PEFT 계열 기법입니다.
학습된 어댑터는 수십 MB에 불과해 작업별로 갈아 끼우기 쉽습니다."""),
("quantization.txt", "경량화", """양자화는 모델 가중치를 저정밀도로 변환해 메모리 사용량을 줄이는 기법입니다.
FP16에서 INT8이나 INT4로 낮추면 모델 크기가 절반 또는 4분의 1로 줄어듭니다.
GPTQ와 AWQ는 대표적인 사후 양자화 방식으로 재학습 없이 적용할 수 있습니다.
GGUF는 CPU 추론에 널리 쓰이는 파일 포맷이며 llama.cpp 계열에서 표준으로 쓰입니다.
비트를 낮출수록 메모리는 줄지만 정확도 손실이 발생할 수 있습니다.
활성값까지 양자화하면 속도 이득이 커지지만 품질 저하 위험도 함께 커집니다.
실무에서는 4bit 양자화가 품질과 비용의 균형점으로 가장 많이 선택됩니다."""),
("distillation.txt", "경량화", """지식 증류는 큰 교사 모델의 출력을 작은 학생 모델이 모방하도록 학습시키는 기법입니다.
정답 레이블 대신 교사의 확률 분포를 학습해 더 풍부한 정보를 전달받습니다.
가지치기는 중요도가 낮은 가중치를 제거해 모델 크기를 줄이는 방법입니다.
구조적 가지치기는 헤드나 층 단위로 제거해 실제 추론 속도까지 개선합니다.
증류와 양자화를 함께 적용하면 경량화 효과가 곱해집니다.
다만 압축을 과하게 하면 희귀 지식과 다단계 추론 능력이 먼저 손상됩니다.
압축 후에는 반드시 원본 모델과 같은 평가셋으로 성능 저하 폭을 측정해야 합니다."""),
("vectordb.txt", "벡터DB", """벡터 데이터베이스는 고차원 벡터를 저장하고 유사도 검색을 수행하는 시스템입니다.
전통적인 관계형 DB가 정확 일치를 찾는다면 벡터 DB는 의미가 가까운 것을 찾습니다.
ChromaDB는 오픈소스 임베디드 벡터 DB로 설치가 간단해 프로토타입에 적합합니다.
FAISS는 Meta가 만든 고속 유사도 검색 라이브러리이며 GPU 가속을 지원합니다.
Pinecone은 인프라 관리가 필요 없는 관리형 클라우드 서비스입니다.
Weaviate는 GraphQL 기반 검색 엔진으로 하이브리드 검색을 기본 제공합니다.
Milvus는 분산 아키텍처로 10억 개 이상의 벡터를 다룰 수 있습니다."""),
("ann_index.txt", "벡터DB", """근사 최근접 이웃 검색은 정확도를 조금 희생하고 속도를 얻는 검색 방식입니다.
HNSW는 계층적 그래프를 타고 이웃을 따라 내려가며 탐색하는 알고리즘입니다.
efSearch 값을 키우면 탐색 폭이 넓어져 정확도가 오르고 속도가 느려집니다.
IVF는 벡터를 클러스터로 나눈 뒤 가까운 클러스터만 탐색하는 방식입니다.
IVF에서는 nprobe 값으로 탐색할 클러스터 수를 정해 정확도를 조절합니다.
PQ는 벡터를 부분 공간으로 쪼개 압축해 메모리를 크게 절감합니다.
완전 탐색은 항상 정확하지만 데이터가 커지면 현실적으로 쓸 수 없습니다."""),
("embedding.txt", "임베딩", """임베딩은 텍스트를 고차원 벡터로 변환해 의미를 수치화하는 과정입니다.
의미가 비슷한 문장은 벡터 공간에서 가까이 위치하도록 학습됩니다.
문장 임베딩 모델은 대조학습으로 유사 쌍은 당기고 비유사 쌍은 밀어냅니다.
한국어에는 KoSimCSE나 multilingual-e5 같은 한국어 지원 모델이 적합합니다.
영어 전용 모델을 한국어에 쓰면 검색 품질이 급격히 무너집니다.
임베딩 차원이 클수록 표현력이 좋지만 저장 공간과 검색 비용이 함께 늘어납니다.
질의용과 문서용 프리픽스를 구분해 넣어야 하는 모델도 있으니 문서를 확인해야 합니다."""),
("similarity.txt", "임베딩", """코사인 유사도는 두 벡터의 각도로 유사도를 측정하며 1에 가까울수록 유사합니다.
내적은 벡터의 크기까지 반영하므로 길이가 다른 벡터에서는 결과가 달라집니다.
유클리드 거리는 값이 작을수록 유사하다는 점에서 방향이 반대입니다.
벡터를 정규화하면 코사인과 내적이 같아지고 L2 거리도 같은 순위를 냅니다.
서로 다른 척도로 만든 점수를 그대로 비교하면 잘못된 결론에 이릅니다.
검색기를 비교할 때는 반드시 같은 거리 척도 위에 올려놓아야 합니다.
BM25 점수와 코사인 점수는 스케일이 달라 단순 합산이 불가능합니다."""),
("prompt_eng.txt", "프롬프트", """프롬프트 엔지니어링은 원하는 출력을 얻기 위한 입력 설계 기술입니다.
같은 모델이라도 입력을 어떻게 쓰느냐에 따라 출력 품질이 크게 달라집니다.
제로샷은 예시 없이 지시만 주고 퓨샷은 몇 개의 입출력 예시를 함께 제공합니다.
예시는 형식을 알려주는 역할이 크므로 원하는 출력 형태를 그대로 보여주는 것이 좋습니다.
시스템 프롬프트로 역할, 제약조건, 응답 형식을 명시적으로 정의합니다.
부정 지시보다 긍정 지시가 더 잘 지켜지는 경향이 있습니다.
출력 형식을 JSON 등으로 고정하면 후처리가 안정적이 됩니다."""),
("cot.txt", "프롬프트", """Chain-of-Thought는 모델이 단계별 추론 과정을 서술하도록 유도하는 기법입니다.
정답만 바로 내놓게 하는 대신 중간 과정을 쓰게 하면 정답률이 오릅니다.
복잡한 수리 문제와 논리 문제에서 특히 효과가 큽니다.
차근차근 생각해보자는 짧은 문구만 넣어도 효과가 나타나는 경우가 있습니다.
Self-Consistency는 여러 추론 경로를 뽑아 다수결로 최종 답을 정합니다.
Tree-of-Thought는 여러 갈래를 탐색하며 유망한 경로를 확장합니다.
추론 과정이 길어지면 토큰 비용과 지연이 함께 늘어난다는 단점이 있습니다."""),
("rl_methods.txt", "강화학습", """강화학습은 에이전트가 환경과 상호작용하며 보상을 최대화하도록 학습하는 방법입니다.
정책은 상태에서 행동을 고르는 규칙이며 학습의 대상이 됩니다.
PPO는 정책이 한 번에 크게 변하지 않도록 제한해 안정적인 업데이트를 보장합니다.
클리핑을 통해 이전 정책과의 비율을 제한하는 것이 PPO의 핵심 아이디어입니다.
GRPO는 DeepSeek이 제안한 효율적인 정책 최적화 방법입니다.
GRPO는 별도의 가치 함수 없이 그룹 내 상대 비교로 이점을 추정해 메모리를 절약합니다.
언어 모델 학습에서는 생성된 응답 전체를 하나의 행동으로 보는 경우가 많습니다."""),
("alignment.txt", "강화학습", """정렬은 모델의 출력을 사람이 바라는 방향으로 맞추는 작업입니다.
RLHF는 인간의 선호도를 보상 모델로 학습한 뒤 강화학습으로 정책에 반영합니다.
먼저 사람이 두 응답 중 나은 것을 고르게 해 선호 데이터를 모읍니다.
그 데이터로 보상 모델을 학습하고 PPO로 정책을 업데이트하는 3단계 구조입니다.
DPO는 별도의 보상 모델 없이 선호 쌍으로 정책을 직접 최적화합니다.
구현이 단순하고 학습이 안정적이어서 최근 널리 쓰입니다.
이런 정렬 기법은 모델의 안전성과 유용성을 높이고 유해한 출력을 줄이는 데 사용됩니다."""),
("rag_basic.txt", "RAG", """RAG는 외부 지식을 검색해 LLM 답변에 근거를 제공하는 기술입니다.
모델의 파라미터에 지식을 넣는 대신 필요할 때 찾아 쓰는 방식입니다.
문서 로딩, 청킹, 임베딩, 검색, 생성의 다섯 단계로 구성됩니다.
파인튜닝 없이 최신 정보를 반영할 수 있다는 것이 가장 큰 장점입니다.
근거 문서를 함께 제시할 수 있어 답변의 출처를 추적할 수 있습니다.
지식이 바뀌면 문서만 교체하면 되므로 유지보수 비용이 낮습니다.
반면 검색이 실패하면 아무리 좋은 LLM도 옳은 답을 낼 수 없습니다."""),
("chunking.txt", "RAG", """청킹은 긴 문서를 검색 단위로 쪼개는 과정이며 RAG 품질의 출발점입니다.
청크가 너무 크면 무관한 내용이 섞여 벡터가 흐려집니다.
너무 작으면 문맥이 끊겨 그 자체로는 의미를 알 수 없는 조각이 됩니다.
오버랩을 두면 경계에서 잘린 문맥을 어느 정도 보완할 수 있습니다.
문단이나 제목 같은 문서 구조를 경계로 삼으면 품질이 좋아집니다.
표와 코드 블록은 중간에서 자르면 의미가 파괴되므로 따로 처리해야 합니다.
적정 크기는 도메인마다 다르므로 실제 질의로 측정해 정하는 것이 맞습니다."""),
("advanced_rag.txt", "RAG", """Advanced RAG는 기본 검색의 약점을 보완하는 기법들의 묶음입니다.
HyDE는 질문으로 가상 답변 문서를 생성한 뒤 그 문서로 검색하는 기법입니다.
질문과 문서의 형태 차이를 줄여 문서 대 문서 비교로 바꾸는 것이 핵심입니다.
리랭킹은 1차 검색 결과를 교차 인코더로 다시 채점해 순서를 바로잡습니다.
교차 인코더는 질문과 문서를 함께 입력해 관련성을 직접 예측합니다.
Parent Document Retriever는 작은 청크로 찾고 큰 부모 청크를 반환합니다.
쿼리 재작성과 컨텍스트 압축도 자주 함께 사용되는 기법입니다."""),
("hybrid_search.txt", "RAG", """하이브리드 검색은 BM25 같은 키워드 검색과 벡터 검색을 결합하는 방식입니다.
BM25는 단어 빈도와 문서 길이를 반영해 점수를 매기는 고전적 검색 알고리즘입니다.
고유명사나 약어, 코드처럼 정확한 토큰 일치가 중요한 질의에서 특히 강합니다.
반대로 표현이 다르면 못 찾는다는 약점이 있어 벡터 검색으로 보완합니다.
RRF는 두 순위 목록을 상호 순위 역수로 융합하는 방법입니다.
점수 스케일이 다른 검색기를 순위만으로 합칠 수 있다는 것이 장점입니다.
가중치는 도메인과 질의 유형에 따라 실측으로 정해야 합니다."""),
("rag_eval.txt", "평가", """RAG 평가는 검색 품질과 생성 품질을 나누어 측정해야 합니다.
검색 품질은 정답 문서를 얼마나 잘 올렸는지를 Hit@k와 MRR로 잽니다.
RAGAS는 RAG 파이프라인을 지표화해 자동 평가하는 프레임워크입니다.
Faithfulness는 답변이 근거 문서에 충실한지를 봅니다.
Answer Relevancy는 답변이 질문에 실제로 대응하는지를 봅니다.
Context Precision과 Context Recall은 검색된 컨텍스트의 질을 따로 측정합니다.
평가 지표를 검색기의 목적함수와 같게 만들면 베이스라인이 항상 이기게 됩니다."""),
("hallucination.txt", "평가", """환각은 모델이 근거 없는 내용을 사실처럼 생성하는 현상입니다.
그럴듯한 문체로 서술되기 때문에 사람이 눈으로 걸러내기 어렵습니다.
검색된 근거를 제시하도록 프롬프트를 설계하면 환각을 줄일 수 있습니다.
근거에 없으면 모른다고 답하게 하는 것이 실무의 기본 전략입니다.
답변의 각 문장을 근거 문서와 대조해 검증하는 후처리도 사용됩니다.
검색이 실패했을 때 억지로 답하지 않게 하는 임계값 설정도 효과적입니다.
환각률은 도메인 전문가가 만든 평가셋으로 주기적으로 측정해야 합니다."""),
("agent.txt", "에이전트", """LLM 에이전트는 도구를 호출하며 여러 단계로 문제를 해결하는 시스템입니다.
한 번의 생성으로 끝내지 않고 관찰과 행동을 반복하는 것이 특징입니다.
ReAct는 추론과 행동을 번갈아 수행하는 대표적인 에이전트 패턴입니다.
계획 수립과 실행을 분리하는 Plan-and-Execute 패턴도 널리 쓰입니다.
LangGraph는 상태 기반 그래프로 에이전트의 흐름을 명시적으로 제어합니다.
반복 횟수 제한과 실패 처리를 두지 않으면 무한 루프에 빠질 수 있습니다.
에이전트는 강력하지만 지연과 비용이 커서 단순 작업에는 과합니다."""),
("tool_use.txt", "에이전트", """함수 호출은 모델이 정해진 스키마에 맞춰 도구 인자를 생성하는 기능입니다.
모델은 도구를 직접 실행하지 않고 어떤 도구를 어떤 인자로 부를지만 출력합니다.
실행은 애플리케이션이 담당하고 그 결과를 다시 모델에 넣어 최종 답변을 만듭니다.
도구 설명과 파라미터 설명을 명확히 쓸수록 호출 정확도가 올라갑니다.
MCP는 모델과 외부 도구를 연결하는 표준 프로토콜입니다.
표준을 쓰면 도구를 여러 애플리케이션에서 재사용할 수 있습니다.
도구 실행에는 권한 검증과 실패 처리를 반드시 함께 설계해야 합니다."""),
]


documents = [
    Document(page_content=content, metadata={"source": src, "topic": topic})
    for src, topic, content in RAW_DOCS
]

# --- 정답 라벨: 질문 → 답이 실제로 들어있는 출처 (사람이 직접 라벨링) ---
# gold를 리스트로 둔 이유: 정답이 여러 문서에 걸칠 수 있기 때문
BENCHMARK = [
    ("소형 언어 모델(sLLM)이 뭐야?",                    ["slm.txt"],            "일반"),
    ("GPU 메모리가 적어도 큰 모델을 학습할 수 있는 방법은?", ["peft.txt"],           "어휘불일치"),
    ("LLM을 더 안전하게 만드는 학습 방법은?",              ["alignment.txt"],      "어휘불일치"),
    ("벡터 검색에 사용할 수 있는 오픈소스 도구들은?",        ["vectordb.txt"],       "일반"),
    ("AI 모델의 출력 품질을 높이는 입력 기법은?",           ["prompt_eng.txt", "cot.txt"], "어휘불일치"),
    ("GRPO가 뭐야?",                                 ["rl_methods.txt"],     "고유명사"),
    ("RTX 4060으로도 학습이 되나?",                     ["peft.txt"],           "고유명사"),
    ("GraphQL을 쓰는 검색 엔진은?",                     ["vectordb.txt"],       "고유명사"),
    ("단계별로 생각하게 시키면 정답률이 오르나?",            ["cot.txt"],            "어휘불일치"),
    ("사람의 선호를 반영해서 모델을 고치는 기법은?",          ["alignment.txt"],      "어휘불일치"),
    ("모델이 없는 사실을 지어내는 걸 뭐라고 하나?",           ["hallucination.txt"],  "일반"),
    ("문서를 얼마나 잘게 쪼개야 하나?",                    ["chunking.txt"],       "일반"),
    ("키워드 검색과 벡터 검색을 같이 쓰는 방법은?",           ["hybrid_search.txt"],  "일반"),
    ("긴 문장을 빠르게 학습시키는 어텐션 최적화는?",          ["attention.txt"],      "어휘불일치"),
    ("CPU에서 모델을 돌릴 때 쓰는 파일 포맷은?",            ["quantization.txt"],   "고유명사"),
    ("검색 결과 순서를 바로잡는 기법은?",                  ["advanced_rag.txt"],   "일반"),
    ("RAG 파이프라인을 수치로 평가하는 도구는?",            ["rag_eval.txt"],       "일반"),
    ("모델이 도구를 직접 호출하게 하려면?",                 ["tool_use.txt"],       "일반"),
    ("근사 최근접 이웃에서 정확도를 조절하는 값은?",          ["ann_index.txt"],      "고유명사"),
    ("한국어 문장 임베딩에 쓸 만한 모델은?",                ["embedding.txt"],      "일반"),
]

test_questions = [q for q, _, _ in BENCHMARK]
gold_sources = {q: set(g) for q, g, _ in BENCHMARK}
question_types = {q: t for q, _, t in BENCHMARK}

# --- 청킹 ---
text_splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=30)
splits = text_splitter.split_documents(documents)

print("📄 벤치마크 준비 완료")
print(f"  문서 {len(documents)}개 → 청크 {len(splits)}개")
print(f"  질문 {len(test_questions)}개 (정답 출처 라벨링 완료)")
print(f"  top-3은 전체 청크의 {3 / len(splits):.0%} — 개선 여지가 남아 있는 규모")

from collections import Counter
print(f"\n  질문 유형 분포: {dict(Counter(question_types.values()))}")

In [ ]:
# 📏 평가 도구 — 정답 라벨 기반 순위 지표 (검색기의 목적함수를 쓰지 않는다)
import time
import numpy as np


def rank_metrics(retrieved_sources, gold):
    """검색된 출처 목록에서 Hit@1 / Hit@3 / RR 을 계산한다.

    Args:
        retrieved_sources: 검색 결과의 source 목록 (순위 순)
        gold: 정답 출처들의 집합
    """
    hit1 = bool(retrieved_sources) and retrieved_sources[0] in gold
    hit3 = any(s in gold for s in retrieved_sources[:3])
    rr = 0.0
    for rank, src in enumerate(retrieved_sources[:3], start=1):
        if src in gold:
            rr = 1.0 / rank      # 첫 정답의 순위만 센다
            break
    return hit1, hit3, rr


def evaluate_retriever(retriever, name, questions=None, k=3, verbose=True):
    """Retriever를 정답 라벨로 평가한다.

    retriever는 .invoke(query) -> list[Document] 를 만족하기만 하면 된다.
    """
    questions = questions or test_questions
    rows, hits1, hits3, rrs, times, ctx_lens = [], [], [], [], [], []

    for q in questions:
        start = time.time()
        docs = retriever.invoke(q)
        elapsed = time.time() - start

        srcs = [d.metadata.get("source", "?") for d in docs[:k]]
        gold = gold_sources[q]
        hit1, hit3, rr = rank_metrics(srcs, gold)

        hits1.append(hit1); hits3.append(hit3); rrs.append(rr)
        times.append(elapsed)
        ctx_lens.append(sum(len(d.page_content) for d in docs[:k]))
        rows.append({"question": q, "sources": srcs, "gold": gold,
                     "hit1": hit1, "hit3": hit3, "rr": rr, "elapsed": elapsed})

    result = {
        "name": name,
        "hit1": float(np.mean(hits1)),
        "hit3": float(np.mean(hits3)),
        "mrr": float(np.mean(rrs)),
        "time": float(np.mean(times)),
        "ctx_len": float(np.mean(ctx_lens)),
        "rows": rows,
    }

    if verbose:
        print(f"\n📊 [{name}]")
        print(f"  Hit@1 {result['hit1']:.0%} | Hit@3 {result['hit3']:.0%} | "
              f"MRR@3 {result['mrr']:.3f} | 평균 {result['time']*1000:.0f}ms | "
              f"평균 컨텍스트 {result['ctx_len']:.0f}자")
        misses = [r for r in rows if not r["hit1"]]
        if misses:
            print(f"  ❌ 1위를 놓친 질문 {len(misses)}개:")
            for r in misses[:5]:
                print(f"     · {r['question']}")
                print(f"       정답={sorted(r['gold'])} / 검색={r['sources']}")
    return result


print("📏 평가 도구 준비 완료")
print("  rank_metrics()      → Hit@1, Hit@3, RR")
print("  evaluate_retriever()→ 지표 + 실패 질문 목록")
print("\n  ⚠️ 이 평가는 임베딩 유사도를 쓰지 않는다.")
print("     사람이 라벨링한 정답 출처와 비교할 뿐이므로, 어떤 검색기에도 유리하지 않다.")

---

## 3️⃣ 기본 RAG 베이스라인

개선 효과를 측정할 기준선을 만듭니다.

### ⚠️ 임베딩 모델은 반드시 한국어를 아는 것으로

한국어 문서에 영어 전용 모델(`all-MiniLM-L6-v2` 등)을 쓰면 검색이 사실상 무작위에 가까워집니다.
이 상태에서는 어떤 Advanced 기법을 얹어도 소용이 없습니다. **기법을 켜기 전에 임베더부터 맞는 것을 씁니다.**

이 노트북의 벤치마크(문서 24개 / 청크 48개 / 질문 20개)에서 **임베딩 모델만 바꿔** 측정한 결과입니다.

| 임베딩 모델 | 한국어 | Hit@1 | Hit@3 | MRR@3 |
|-------------|--------|-------|-------|-------|
| `all-MiniLM-L6-v2` | ✗ 영어 전용 | **10%** | 25% | 0.175 |
| `BM-K/KoSimCSE-roberta-multitask` | ✓ 한국어 | **65%** | 90% | 0.775 |

문서가 24개이므로 **무작위로 찍어도 Hit@1이 약 4%** 입니다.
영어 모델의 10%는 그보다 조금 나은 정도, 즉 **사실상 검색이 동작하지 않는 상태**입니다.
이 위에 HyDE든 리랭킹이든 얹어봐야 아무 의미가 없습니다.

> 🔑 **Advanced RAG를 논하기 전에 임베더부터 맞는 것을 쓰세요.**
> 기법 하나 추가해서 얻는 개선보다, 임베더를 바꿔서 얻는 개선(MRR 0.175 → 0.775, 4.4배)이 훨씬 큽니다.

Session 03에서 쓴 것과 같은 KoSimCSE 모델을 사용합니다.


In [ ]:
# 🔧 임베딩 · 벡터스토어 · LLM 설정
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# ⚠️ 한국어 코퍼스에는 반드시 한국어를 아는 임베딩 모델을 쓴다.
#    all-MiniLM-L6-v2 같은 영어 전용 모델을 쓰면 검색이 거의 무작위가 되고,
#    그 위에 어떤 Advanced 기법을 얹어도 효과가 나타나지 않는다.
EMBEDDING_MODEL = "BM-K/KoSimCSE-roberta-multitask"   # Session 03과 동일

embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
    model_kwargs={"device": DEVICE},
    encode_kwargs={"normalize_embeddings": True},   # 정규화 → 코사인=내적
)
print(f"✅ 임베딩 모델: {EMBEDDING_MODEL}")

# 벡터스토어 (코사인 거리)
vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=embeddings,
    collection_name="advanced_rag_demo",
    collection_metadata={"hnsw:space": "cosine"},
)
print(f"✅ 벡터스토어 구축 완료 ({vectorstore._collection.count()}개 청크)")

# 기본 Retriever
base_retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# LLM (HyDE 섹션에서만 사용)
USE_OLLAMA = False
llm = None
try:
    from dotenv import load_dotenv, find_dotenv
    load_dotenv(find_dotenv(usecwd=True))
except Exception:
    pass

if USE_OLLAMA:
    from langchain_community.llms import Ollama
    llm = Ollama(model="qwen2.5:1.5b", temperature=0, num_ctx=2048)
    print("✅ Ollama LLM 설정 완료")
elif os.getenv("OPENAI_API_KEY"):
    from langchain_openai import ChatOpenAI
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    print("✅ OpenAI LLM 설정 완료 (gpt-4o-mini)")
else:
    print("⚠️ OPENAI_API_KEY 없음 — HyDE 섹션은 건너뜁니다 (나머지는 정상 동작)")

In [ ]:
# 📊 기본 RAG 베이스라인 측정
base_result = evaluate_retriever(base_retriever, "기본 RAG")

# 이 베이스라인이 이후 모든 기법의 비교 기준이 된다
print("\n" + "=" * 70)
print("이 숫자가 기준선입니다. 이제 기법을 하나씩 얹으며 이 값을 넘는지 봅니다.")
print("=" * 70)

---

## 4️⃣ HyDE (Hypothetical Document Embeddings)

**HyDE**는 질문을 가상의 답변 문서로 변환한 후 검색하는 기법입니다.

### 🔑 HyDE의 핵심 아이디어
```
기본 RAG:  질문 임베딩 → 문서 임베딩과 비교
HyDE:      질문 → LLM으로 가상 답변 생성 → 가상 답변 임베딩 → 문서 임베딩과 비교
```

- 🔹 질문과 문서의 **형태 차이**를 해소 (질문은 짧고 의문형, 문서는 길고 서술형)
- 🔹 "질문 ↔ 문서"를 "문서 ↔ 문서" 유사도 비교로 변환
- 🔹 LLM의 지식을 검색 쿼리에 주입

### ⚠️ 이 기법의 전제와 비용

HyDE는 **LLM이 그 도메인을 알고 있을 때만** 제대로 작동합니다.
LLM이 모르는 용어라면 그럴듯한 **다른 분야의 문서를 지어내고**, 그 오염된 벡터로 검색하게 됩니다.
아래 셀에서 생성된 가상 문서를 직접 눈으로 확인해 보세요.
(실제로 "GRPO"를 물으면 gpt-4o-mini는 이것을 **공급망 관리 문서**로 착각합니다.)

그리고 **비용이 압도적으로 큽니다.** 질의마다 LLM을 한 번씩 호출하므로,
이 벤치마크에서 기본 RAG가 약 15ms인 반면 HyDE는 **약 1,400ms — 100배 가까이** 걸립니다.

> ⚠️ **HyDE의 결과는 실행할 때마다 달라집니다.** `temperature=0`이어도 LLM 생성은 완전히 결정적이지 않아,
> 같은 코드를 두 번 돌리면 MRR이 **-4% ~ +2%** 사이에서 오갑니다.
> **실행 간 변동폭이 개선 폭보다 크다면, 그 개선은 실재하지 않는 것**입니다.
> 여러분이 이 셀을 돌렸을 때 나오는 숫자도 아래 해설과 다를 수 있습니다 — 그것이 이 기법에 대한 결론입니다.

> 💡 이 섹션만 OpenAI API 키가 필요합니다. 키가 없으면 셀이 안내 메시지를 출력하고 넘어갑니다.


In [ ]:
# 🔮 HyDE Retriever 구현
from langchain.prompts import PromptTemplate

hyde_prompt = PromptTemplate(
    template="""다음 질문에 대한 답변을 기술 문서의 한 문단처럼 3문장 이내로 작성하세요.
사실 여부는 중요하지 않습니다. 질문에 등장하지 않은 관련 전문 용어를 포함하세요.

질문: {question}

답변 문단:""",
    input_variables=["question"],
)


class HyDERetriever:
    """HyDE: 질문 → LLM이 가상 답변 생성 → 그 답변으로 벡터 검색"""

    def __init__(self, llm, vectorstore, k=3):
        self.llm = llm
        self.vectorstore = vectorstore
        self.k = k
        self.last_hypothetical = None      # 관찰용: 마지막으로 생성한 가상 문서

    def invoke(self, query):
        # 1. LLM으로 가상 답변 문서 생성
        out = self.llm.invoke(hyde_prompt.format(question=query))
        hypothetical = out if isinstance(out, str) else out.content
        self.last_hypothetical = hypothetical.strip()

        # 2. 질문이 아니라 '가상 문서'로 검색한다 — 이것이 HyDE의 전부
        return self.vectorstore.similarity_search(self.last_hypothetical, k=self.k)

    def get_relevant_documents(self, query):
        return self.invoke(query)


if llm is None:
    hyde_retriever = None
    print("⚠️ LLM이 없어 HyDE를 건너뜁니다.")
else:
    hyde_retriever = HyDERetriever(llm, vectorstore, k=3)
    print("✅ HyDE Retriever 생성 완료")
    print("  📊 동작: 질문 → LLM(가상 문서 생성) → 벡터 검색")

In [ ]:
# 🔍 HyDE가 만드는 가상 문서를 직접 확인 — 이 기법의 성패가 여기서 갈린다
if hyde_retriever is None:
    print("⚠️ LLM 없음 — 건너뜁니다.")
else:
    for q in ["GRPO가 뭐야?", "CPU에서 모델을 돌릴 때 쓰는 파일 포맷은?"]:
        docs = hyde_retriever.invoke(q)
        print(f"❓ 질문: {q}")
        print(f"🔮 LLM이 지어낸 가상 문서:")
        print(f"   {hyde_retriever.last_hypothetical[:220]}...")
        print(f"📄 그 문서로 검색한 결과: {[d.metadata['source'] for d in docs]}")
        print(f"✅ 정답: {sorted(gold_sources[q])}")
        print("-" * 70)

    print("\n💡 LLM이 도메인을 모르면 엉뚱한 분야의 문서를 지어냅니다.")
    print("   그 오염된 벡터로 검색하므로 결과가 함께 오염됩니다.")

In [ ]:
# 📊 HyDE 평가
if hyde_retriever is None:
    hyde_result = None
    print("⚠️ LLM 없음 — HyDE 평가를 건너뜁니다.")
else:
    hyde_result = evaluate_retriever(hyde_retriever, "HyDE")
    print(f"\n  기본 RAG 대비 Hit@1: {base_result['hit1']:.0%} → {hyde_result['hit1']:.0%}")

---

## 5️⃣ Reranking (재순위화)

**Reranking**은 초기 검색 결과를 더 정교한 모델로 다시 순위를 매기는 기법입니다.

### 🔑 Reranking의 원리
```
1단계: 빠른 검색 (Bi-encoder)   → 후보 문서 N개 (넓게)
2단계: 정밀 평가 (Cross-encoder) → 재순위화 → 상위 K개 (좁게)
```

- 🔹 **Bi-encoder**: 질문과 문서를 **따로** 인코딩해 벡터 하나씩 만든 뒤 내적. 미리 계산해 둘 수 있어 빠르지만 덜 정확
- 🔹 **Cross-encoder**: 질문과 문서를 **하나로 붙여** 모델에 넣고 관련성 점수를 직접 출력. 느리지만 훨씬 정확

Cross-encoder는 모든 (질문, 문서) 쌍마다 모델을 돌려야 해서 전체 코퍼스에는 쓸 수 없습니다.
그래서 **1단계로 후보를 좁힌 뒤 2단계로 정밀 채점**하는 2단계 구조를 씁니다.

### ⚠️ 리랭커도 한국어를 알아야 합니다

한국어 코퍼스에서 리랭커 모델만 바꿔 측정한 결과입니다.

임베더를 KoSimCSE로 고정하고 **리랭커 모델만 바꿔** 측정한 결과입니다.

| Cross-encoder 모델 | 언어 | Hit@1 | Hit@3 | MRR@3 |
|--------------------|------|-------|-------|-------|
| 리랭킹 없음 (기본 RAG) | — | 65% | 90% | 0.775 |
| `cross-encoder/ms-marco-MiniLM-L-6-v2` | 영어 전용 | **45%** | 65% | 0.550 |
| `BAAI/bge-reranker-base` | 다국어(소형) | 50% | 60% | 0.550 |
| `BAAI/bge-reranker-v2-m3` | 다국어 | **95%** | 95% | **0.950** |

**영어 리랭커를 쓰면 기본 RAG보다 오히려 나빠집니다.** 재순위화가 아니라 무작위 섞기에 가까워지기 때문입니다.
같은 "리랭킹 기법"인데 **모델 선택 하나로 45%와 95%가 갈립니다.**
기법이 효과가 없어 보일 때 가장 먼저 의심할 것은 기법이 아니라 모델입니다.


In [ ]:
# 🎯 Reranking Retriever 구현
from sentence_transformers import CrossEncoder

# ⚠️ 리랭커도 한국어를 알아야 한다.
#    cross-encoder/ms-marco-MiniLM-L-6-v2 는 영어 전용이라 한국어에서는
#    기본 RAG보다 오히려 나쁘다 (과제 2번에서 직접 확인).
RERANKER_MODEL = "BAAI/bge-reranker-v2-m3"   # 다국어 cross-encoder (약 2.2GB)

print(f"🔧 Cross-encoder 로딩 중: {RERANKER_MODEL}")
print("   (최초 실행 시 다운로드에 시간이 걸립니다)")
reranker_model = CrossEncoder(RERANKER_MODEL, max_length=512, device=DEVICE)
print("✅ Reranker 로딩 완료")


class RerankingRetriever:
    """2단계 검색: 넓게 뽑고(bi-encoder) → 정밀하게 재채점(cross-encoder)"""

    def __init__(self, vectorstore, reranker, top_k=3, fetch_k=10):
        self.vectorstore = vectorstore
        self.reranker = reranker
        self.top_k = top_k
        self.fetch_k = fetch_k          # 1단계에서 뽑을 후보 수

    def invoke(self, query):
        # 1단계: 빠른 벡터 검색으로 후보를 넓게 확보
        candidates = self.vectorstore.similarity_search(query, k=self.fetch_k)
        if not candidates:
            return []

        # 2단계: (질문, 문서) 쌍마다 cross-encoder로 관련성 점수 산출
        pairs = [[query, doc.page_content] for doc in candidates]
        scores = self.reranker.predict(pairs)

        ranked = sorted(zip(candidates, scores), key=lambda x: x[1], reverse=True)
        return [doc for doc, _ in ranked[: self.top_k]]

    def get_relevant_documents(self, query):
        return self.invoke(query)


reranking_retriever = RerankingRetriever(vectorstore, reranker_model, top_k=3, fetch_k=10)
print("✅ Reranking Retriever 생성 완료")
print("  📊 동작: 벡터 검색 10개 → cross-encoder 재채점 → 상위 3개")

# 리랭킹이 의미가 있으려면 1단계 후보 안에 정답이 들어 있어야 한다. 먼저 그것부터 확인한다.
pool_hits = 0
for q in test_questions:
    pool = vectorstore.similarity_search(q, k=10)
    if any(d.metadata["source"] in gold_sources[q] for d in pool):
        pool_hits += 1
print(f"\n🔍 1단계 후보(top-10)에 정답이 포함된 비율: {pool_hits}/{len(test_questions)} "
      f"= {pool_hits/len(test_questions):.0%}")
print("   → 이 값이 리랭킹으로 도달 가능한 Hit@1의 상한이다.")
print("     정답은 이미 후보 안에 있다. 문제는 '순서'였다.")

In [ ]:
# 📊 Reranking 평가
reranking_result = evaluate_retriever(reranking_retriever, "Reranking")
print(f"\n  기본 RAG 대비 Hit@1: {base_result['hit1']:.0%} → {reranking_result['hit1']:.0%}")
print(f"  기본 RAG 대비 MRR@3: {base_result['mrr']:.3f} → {reranking_result['mrr']:.3f}")

---

## 6️⃣ Ensemble Retriever (BM25 + 시맨틱)

**Ensemble Retriever**는 키워드 기반 검색(BM25)과 의미 기반 검색(시맨틱)을 결합합니다.

### 🔑 BM25 vs 시맨틱 검색
| 특성 | BM25 | 시맨틱 검색 |
|------|------|------------|
| 매칭 방식 | 키워드 일치 | 의미 유사도 |
| 장점 | 정확한 키워드 매칭 (고유명사, 코드, 약어) | 동의어/유사 표현 |
| 단점 | 동의어 놓침 | 희귀 고유명사에 약함 |
| 속도 | 빠름 | 임베딩 필요 |

### 🔗 어떻게 합치나 — RRF (Reciprocal Rank Fusion)

두 검색기가 내놓은 **순위 목록**을 점수 대신 순위로 융합합니다.
BM25 점수와 코사인 유사도는 스케일이 달라 그냥 더할 수 없기 때문입니다.

```
RRF 점수(문서) = Σ  가중치 / (k + 그 검색기에서의 순위)      (보통 k=60)
```

### ⚠️ 가중치를 0.5 / 0.5 로 두지 마세요 — 동점의 함정

두 검색기의 가중치가 같으면 **RRF 점수가 정확히 같아지는 동점이 대량 발생**합니다.

```
BM25  1위 : 0.5 / (60 + 1) = 0.008197
dense 1위 : 0.5 / (60 + 1) = 0.008197     ← 소수점까지 완전히 동일
```

파이썬 `sorted()`는 안정 정렬이고 LangChain은 `retrievers=[...]`에 넘긴 **순서대로**
문서를 이어붙입니다. 따라서 동점이면 **리스트에 먼저 적은 검색기가 전부 이깁니다.**

실제로 이 벤치마크에서 `weights=[0.5, 0.5]`로 두면 **Ensemble의 1위가 BM25의 1위와
20개 중 18개에서 같았습니다.** 섞인 게 아니라 BM25가 통째로 채택된 것입니다.

| 구성 | Hit@1 | 일반 | 고유명사 | 어휘불일치 |
|------|-------|------|----------|------------|
| dense 단독 | 65% | 100% | 20% | 50% |
| BM25 단독 | 60% | 89% | 40% | 33% |
| `[bm25, dense]` w=0.5/0.5 | **60%** | 89% | 40% | 33% |
| `[dense, bm25]` w=0.5/0.5 **(순서만 바꿈)** | **75%** | 100% | 60% | 50% |
| `[bm25, dense]` w=**0.4/0.6** (이 노트북) | **75%** | 100% | 60% | 50% |

**코드에서 바뀐 것이 리스트 순서뿐인데 Hit@1이 60%와 75%로 갈립니다.**
해결책은 간단합니다 — **가중치를 서로 다르게 주어 동점 자체를 없애면** 됩니다.

이 노트북은 `weights=[0.4, 0.6]`(BM25 40% / 시맨틱 60%)을 씁니다.
아래에서 RRF를 직접 구현해 라이브러리와 결과가 같은지 확인하고, 이 함정도 재현해 봅니다.


In [ ]:
# 🔗 Ensemble Retriever (BM25 + 시맨틱)
from langchain_community.retrievers import BM25Retriever
from langchain.retrievers import EnsembleRetriever

# BM25 — 키워드(토큰) 일치 기반
bm25_retriever = BM25Retriever.from_documents(splits)
bm25_retriever.k = 3
print("✅ BM25 Retriever 생성 완료")

# ⚠️ 가중치를 0.5 / 0.5 로 두면 안 된다. 아래 셀에서 이유를 직접 확인한다.
#    RRF 점수가 정확히 같아지는 '동점'이 대량 발생하고, 동점일 때는
#    retrievers 리스트에 먼저 적은 검색기가 무조건 이긴다.
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, base_retriever],
    weights=[0.4, 0.6],          # BM25 40% + 시맨틱 60% — 동점을 없앤다
)
print("✅ Ensemble Retriever 생성 완료 (BM25 40% + 시맨틱 60%, RRF 융합)")

In [ ]:
# 🔬 RRF를 직접 구현해 본다 — EnsembleRetriever가 내부에서 하는 일
# LangChain은 weighted_reciprocal_rank() 안에서 아래와 똑같은 계산을 한다.
#   langchain/retrievers/ensemble.py :
#       rrf_score[doc.page_content] += weight / (rank + self.c)

RRF_C = 60      # LangChain EnsembleRetriever의 기본값 (c 파라미터)


def manual_rrf(query, retrievers, weights, c=RRF_C, top_k=3, explain=False):
    """가중 RRF 융합을 직접 계산한다.

    점수를 쓰지 않고 '순위'만 쓴다는 것이 핵심이다.
    BM25 점수(상한 없음)와 코사인 유사도(0~1)는 스케일이 달라 더할 수 없기 때문이다.
    """
    scores, seen, trace = {}, {}, {}
    for retriever, w, label in zip(retrievers, weights, ["BM25", "dense"]):
        for rank, doc in enumerate(retriever.invoke(query), start=1):
            key = doc.page_content                  # LangChain과 동일한 기준으로 문서 식별
            contrib = w / (c + rank)
            scores[key] = scores.get(key, 0.0) + contrib     # ← 두 목록에 다 나오면 '누적'
            seen[key] = doc
            trace.setdefault(key, []).append(f"{label}{rank}위 {w}/({c}+{rank})={contrib:.6f}")

    ranked = sorted(scores.items(), key=lambda x: -x[1])
    if explain:
        for key, sc in ranked:
            print(f"  {seen[key].metadata['source']:<20}{sc:.6f}   {'  +  '.join(trace[key])}")
    return [seen[k] for k, _ in ranked[:top_k]]


# --- 계산 과정 출력 ---
demo_q = "근사 최근접 이웃에서 정확도를 조절하는 값은?"
print(f"질문: {demo_q}")
print(f"정답 출처: {sorted(gold_sources[demo_q])}\n")

print("[1단계] 각 검색기의 순위 목록")
b_list = [d.metadata["source"] for d in bm25_retriever.invoke(demo_q)]
d_list = [d.metadata["source"] for d in base_retriever.invoke(demo_q)]
print(f"  {'순위':<6}{'BM25':<22}{'시맨틱(dense)':<22}")
for i in range(max(len(b_list), len(d_list))):
    print(f"  {i+1:<6}{b_list[i] if i < len(b_list) else '':<22}"
          f"{d_list[i] if i < len(d_list) else '':<22}")

print(f"\n[2단계] RRF 융합 (c={RRF_C}, 가중치 BM25 0.4 / dense 0.6)")
fused = manual_rrf(demo_q, [bm25_retriever, base_retriever], [0.4, 0.6], explain=True)

print(f"\n[3단계] 최종 top-3: {[d.metadata['source'] for d in fused]}")
lib = [d.metadata["source"] for d in ensemble_retriever.invoke(demo_q)][:3]
print(f"        EnsembleRetriever 결과: {lib}")
print(f"        일치 여부: {'✅ 같다' if [d.metadata['source'] for d in fused] == lib else '❌ 다르다'}")

In [ ]:
# ⚠️ 동점의 함정 — 가중치가 같으면 '리스트 순서'가 결과를 정한다
# 두 검색기의 가중치가 같으면 1위끼리 RRF 점수가 정확히 같아진다.
#     BM25  1위 : 0.5 / (60+1) = 0.008197
#     dense 1위 : 0.5 / (60+1) = 0.008197   ← 완전히 동일
# 파이썬 sorted()는 안정 정렬이라 동점이면 먼저 들어온 쪽이 앞선다.
# LangChain은 retrievers 리스트 순서대로 문서를 이어붙이므로,
# '먼저 적은 검색기'가 동점 승부를 전부 가져간다.

import numpy as np


def quick_hit1(retriever):
    return np.mean([retriever.invoke(q)[0].metadata["source"] in gold_sources[q]
                    for q in test_questions])


def hit1_by_type(retriever):
    out = {}
    for t in ["일반", "고유명사", "어휘불일치"]:
        sel = [q for q in test_questions if question_types[q] == t]
        out[t] = np.mean([retriever.invoke(q)[0].metadata["source"] in gold_sources[q]
                          for q in sel])
    return out


configs = [
    ("dense 단독",                        base_retriever),
    ("BM25 단독",                         bm25_retriever),
    ("[bm25, dense] w=0.5/0.5",          EnsembleRetriever(
        retrievers=[bm25_retriever, base_retriever], weights=[0.5, 0.5])),
    ("[dense, bm25] w=0.5/0.5 (순서만 바꿈)", EnsembleRetriever(
        retrievers=[base_retriever, bm25_retriever], weights=[0.5, 0.5])),
    ("[bm25, dense] w=0.4/0.6 (이 노트북)",  ensemble_retriever),
]

print(f"{'구성':<40}{'Hit@1':>8}{'일반':>8}{'고유명사':>9}{'어휘불일치':>10}")
print("-" * 76)
for name, ret in configs:
    t = hit1_by_type(ret)
    print(f"{name:<38}{quick_hit1(ret):>8.0%}{t['일반']:>8.0%}"
          f"{t['고유명사']:>8.0%}{t['어휘불일치']:>10.0%}")

print("""
💡 가중치 0.5/0.5 두 줄을 비교하세요. 코드에서 바뀐 것은 리스트 순서뿐인데
   Hit@1이 크게 달라집니다. 앙상블이 '작동하지 않는다'고 결론 내리기 전에
   동점 때문에 한쪽 검색기가 통째로 채택되고 있지는 않은지 확인해야 합니다.

   해결책은 간단합니다 — 가중치를 서로 다르게 주어 동점 자체를 없애면 됩니다.""")

In [ ]:
# 📊 BM25 단독 / Ensemble 평가 — 섞을 값어치가 있는지부터 확인한다
bm25_result = evaluate_retriever(bm25_retriever, "BM25 단독", verbose=False)
print(f"📊 [BM25 단독]  Hit@1 {bm25_result['hit1']:.0%} | Hit@3 {bm25_result['hit3']:.0%} | "
      f"MRR@3 {bm25_result['mrr']:.3f}")
print(f"📊 [기본 RAG]   Hit@1 {base_result['hit1']:.0%} | Hit@3 {base_result['hit3']:.0%} | "
      f"MRR@3 {base_result['mrr']:.3f}")

ensemble_result = evaluate_retriever(ensemble_retriever, "Ensemble")

# 질문 유형별로 쪼개 보면 BM25의 진짜 가치가 드러난다
print("\n" + "=" * 70)
print("질문 유형별 Hit@1 — 평균만 보면 놓치는 것")
print("=" * 70)
types = ["일반", "고유명사", "어휘불일치"]
print(f"{'유형':<12}{'기본 RAG':>10}{'BM25':>10}{'Ensemble':>10}   질문 수")
print("-" * 56)
for t in types:
    idx = [i for i, q in enumerate(test_questions) if question_types[q] == t]
    def h1(res):
        return np.mean([res["rows"][i]["hit1"] for i in idx])
    print(f"{t:<10}{h1(base_result):>10.0%}{h1(bm25_result):>10.0%}"
          f"{h1(ensemble_result):>10.0%}   {len(idx)}개")

print("\n💡 dense는 고유명사에 약하고, BM25는 어휘불일치에 약합니다 — 약점이 서로 반대입니다.")
print("   앙상블이 성립하는 이유가 이것이고, 실제로 고유명사 정확도가 크게 올라갑니다.")
print("   단, 가중치를 잘못 주면 이 이득이 통째로 사라집니다 (앞 셀의 동점 실험 참고).")

---

## 7️⃣ Parent Document Retriever

**Parent Document Retriever**는 작은 청크로 검색하되, 큰 문맥(부모 문서)을 반환합니다.

### 🔑 핵심 아이디어
```
문서 → 큰 청크(부모) → 작은 청크(자식)
                        ↓
검색: 작은 청크로 정밀 매칭
반환: 큰 청크(부모)로 풍부한 문맥
```

- 🔹 작은 청크: 검색 정확도 높음 (노이즈가 적어 벡터가 선명함)
- 🔹 큰 청크: LLM에게 더 풍부한 문맥 제공

### ⚠️ 순위 지표로는 이 기법의 장점이 잡히지 않습니다

Parent Document Retriever가 개선하는 것은 **"찾은 뒤에 무엇을 건네주는가"** 이지
**"무엇을 찾는가"** 가 아닙니다. Hit@k와 MRR은 후자만 재는 지표이므로,
이 기법은 이 벤치마크에서 사실상 본전(MRR +2%)으로 나옵니다.
반면 **반환 컨텍스트는 약 1.5배**로 늘어납니다 — 그 이득은 순위 지표에 전혀 반영되지 않습니다.

**지표가 기법을 담지 못하는 경우**이며, 이럴 때 필요한 것이 답변 품질 평가(Session 08의 RAGAS)입니다.
아래에서는 순위 지표와 함께 **반환된 컨텍스트의 길이**를 같이 측정해 실제 차이를 봅니다.


In [ ]:
# 📚 Parent Document Retriever
from langchain.retrievers import ParentDocumentRetriever
from langchain.storage import InMemoryStore

parent_splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=50)
child_splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=20)

docstore = InMemoryStore()
parent_vectorstore = Chroma(
    collection_name="parent_doc_demo",
    embedding_function=embeddings,
    collection_metadata={"hnsw:space": "cosine"},
)

parent_retriever = ParentDocumentRetriever(
    vectorstore=parent_vectorstore,
    docstore=docstore,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
    search_kwargs={"k": 3},
)
parent_retriever.add_documents(documents)

print("✅ Parent Document Retriever 생성 완료")
print("  📊 자식 청크 100자로 검색 → 부모 청크 400자를 반환")

# 📊 평가 — 순위 지표와 함께 '반환된 컨텍스트 길이'를 같이 본다
parent_result = evaluate_retriever(parent_retriever, "Parent Doc")

print("\n" + "=" * 70)
print("이 기법의 실제 효과는 순위가 아니라 '반환 컨텍스트의 양'에 있다")
print("=" * 70)
print(f"  기본 RAG        평균 컨텍스트 {base_result['ctx_len']:.0f}자 | Hit@3 {base_result['hit3']:.0%}")
print(f"  Parent Document 평균 컨텍스트 {parent_result['ctx_len']:.0f}자 | Hit@3 {parent_result['hit3']:.0%}")
print(f"\n  → 컨텍스트는 {parent_result['ctx_len']/max(base_result['ctx_len'],1):.1f}배로 늘었지만,")
print("    Hit@k는 그 이득을 표현하지 못한다. 지표가 기법을 담지 못하는 경우다.")

---

## 8️⃣ 종합 성능 비교

같은 코퍼스, 같은 질문 20개, 같은 정답 라벨로 모든 기법을 한 표에 놓습니다.


In [ ]:
# 📊 종합 성능 비교 — 같은 코퍼스, 같은 질문 20개, 같은 정답 라벨
all_results = [r for r in [
    base_result, hyde_result, reranking_result, ensemble_result, parent_result
] if r is not None]

print("=" * 84)
print(f"Advanced RAG 종합 비교 (문서 {len(documents)}개 / 청크 {len(splits)}개 / 질문 {len(test_questions)}개)")
print("=" * 84)
print(f"{'방법':<22}{'Hit@1':>9}{'Hit@3':>9}{'MRR@3':>9}{'지연(ms)':>11}{'컨텍스트':>10}{'baseline 대비':>14}")
print("-" * 84)

for r in all_results:
    delta = (r["mrr"] / base_result["mrr"] - 1) * 100 if base_result["mrr"] else 0
    tag = "기준" if r is base_result else f"{delta:+.0f}%"
    bar = "█" * int(r["mrr"] * 20)
    print(f"{r['name']:<20}{r['hit1']:>9.0%}{r['hit3']:>9.0%}{r['mrr']:>9.3f}"
          f"{r['time']*1000:>11.0f}{r['ctx_len']:>9.0f}자{tag:>12}  {bar}")
print("=" * 84)

# 질문별 승패표 — 평균 뒤에 가려진 것을 본다
print("\n질문별 1위 적중 (O = 정답이 1위)")
print("-" * 84)
names = [r["name"] for r in all_results]
print(f"{'질문':<40}{'유형':<12}" + "".join(f"{n:>14}" for n in names))
for i, q in enumerate(test_questions):
    marks = "".join(f"{('O' if r['rows'][i]['hit1'] else '·'):>14}" for r in all_results)
    print(f"{q[:38]:<40}{question_types[q]:<10}{marks}")

# 승자 요약
best = max(all_results, key=lambda r: r["mrr"])
print(f"\n🏆 MRR@3 최고: {best['name']} ({best['mrr']:.3f}, "
      f"기본 RAG 대비 {(best['mrr']/base_result['mrr']-1)*100:+.0f}%)")
losers = [r["name"] for r in all_results if r["mrr"] < base_result["mrr"]]
if losers:
    print(f"⚠️ 기본 RAG보다 낮은 기법: {', '.join(losers)}")
    print("   → 기법이 나쁜 게 아니라, 이 데이터/모델 조건에 맞지 않는다는 뜻이다.")

### 📌 결과 해석 — 무엇이 이기고 무엇이 졌나

실측 결과 (문서 24개 / 청크 48개 / 질문 20개, 정답 라벨 기준)

| 방법 | Hit@1 | MRR@3 | 지연 | baseline 대비 | 진짜 원인 |
|------|-------|-------|------|---------------|-----------|
| **Reranking** | 65% → **95%** | 0.775 → **0.950** | 14ms → 69ms | 🥇 **+23%** | 1차 검색이 정답을 top-10 안에는 **항상**(20/20) 넣어준다. 문제는 순서였고 cross-encoder가 그것을 바로잡는다 |
| **HyDE** | 65% → 60~70% | 0.775 → 0.742~0.792 | 15ms → **~1,400ms** | **-4% ~ +2%** (실행마다 변동) | 개선이 실행 간 잡음보다 작다 = **개선 없음**. 비용은 100배. LLM이 모르는 용어("GRPO")는 엉뚱한 분야로 지어낸다 |
| **Parent Document** | 65% → 70% | 0.775 → 0.792 | 11ms | +2% | 검색은 자식 청크로 하므로 순위가 거의 그대로. 이득(컨텍스트 1.5배)이 **Hit@k로는 안 잡힌다** |
| **Ensemble** | 65% → **75%** | 0.775 → **0.825** | 11ms | 🥈 **+6%** | dense가 약한 고유명사를 BM25가 메운다. **단 `weights=[0.5,0.5]`로 두면 -13%로 뒤집힌다** — 동점 때문에 BM25가 통째로 채택되기 때문 |

### 🔍 평균 뒤에 가려진 것 — 질문 유형별 Hit@1

| 유형 | 기본 RAG | BM25 | Ensemble (0.4/0.6) | 해석 |
|------|----------|------|--------------------|------|
| 일반 (9개) | 100% | 89% | **100%** | dense가 이미 완벽 — 앙상블이 깎아먹지 않았다 |
| 고유명사 (5개) | **20%** | 40% | **60%** | dense의 약점을 BM25가 메운다. 세 배 |
| 어휘불일치 (6개) | 50% | 33% | **50%** | BM25의 약점을 dense가 지켜준다 |

**이것이 앙상블이 제대로 작동할 때의 모습입니다.** 잘하는 유형은 그대로 지키고
못하는 유형만 끌어올립니다. 전체 이득(+6%)이 크지 않아 보이지만, **고유명사 질의가
많은 도메인**(제품 코드, 사번, 법조문 번호, API 이름)이라면 훨씬 커집니다.
**앙상블의 가치는 평균이 아니라 "어떤 실패 유형을 메우는가"로 판단해야 합니다.**

### 🎯 핵심 교훈

1. **기법보다 모델 선택이 먼저다**
   임베더를 영어 모델로 바꾸면 MRR이 0.775 → 0.175로 무너집니다. 어떤 기법도 이걸 되돌리지 못합니다.
   리랭킹도 마찬가지로, 영어 모델은 45% / 다국어 모델은 95%입니다.
   **기법이 안 듣는다고 느껴지면 기법이 아니라 모델을 의심하세요.**

2. **설정 하나가 결론을 뒤집는다**
   Ensemble은 `weights=[0.5, 0.5]`에서 -13%, `[0.4, 0.6]`에서 +6%입니다.
   가중치가 같으면 RRF 동점이 생기고, 동점은 리스트에 먼저 적은 검색기가 전부 가져갑니다.
   **기법이 안 듣는다고 판단하기 전에 그 기법의 기본 설정부터 의심하세요.**

3. **잘못 측정하면 개선이 사라진다**
   검색기의 목적함수(코사인 유사도)를 평가 지표로 쓰면 기본 RAG가 **수학적으로 항상 이깁니다.**
   정답 라벨 20개를 만드는 데 드는 30분이, 기법 도입 판단의 전부를 좌우합니다.

4. **지표가 기법을 담는지 확인하라**
   Parent Document의 이득(컨텍스트 1.5배)은 Hit@k에 나타나지 않습니다.
   지표를 바꾸지 않으면 이 기법은 영원히 "효과 없음"으로 남습니다.

5. **평균 하나로 판단하지 마라**
   Ensemble의 전체 이득은 +6%지만, 고유명사 질의만 보면 20% → 60%로 세 배입니다.
   **실패한 질문을 유형별로 쪼개 보는 것**이 다음에 어떤 기법을 붙일지 알려줍니다.

6. **개선 폭이 실행 간 변동폭보다 큰지 확인하라**
   HyDE는 돌릴 때마다 -4% ~ +2%를 오갑니다. 한 번 돌려 +2%가 나왔다고 "효과 있음"이라 적으면 안 됩니다.
   게다가 지연은 100배입니다. 같은 예산이면 리랭킹이 압도적으로 낫습니다.

7. **최종 판단은 답변 품질로 한다**
   검색 지표는 중간 지표입니다. 최종적으로는 답변의 정확성(LLM-as-a-Judge, RAGAS)으로 평가해야 하며,
   그것이 Session 08(`08_rag_evaluation`)의 주제입니다.


In [ ]:
# 💡 기법별 적합한 사용 상황 정리
print("💡 Advanced RAG 기법 선택 가이드")
print("=" * 72)

guide = [
    {
        "method": "Reranking",
        "best_for": "1차 검색이 정답을 top-N에는 넣는데 순서가 틀릴 때",
        "cost": "Cross-encoder 추론 (후보 수에 비례)",
        "caution": "반드시 다국어/한국어 리랭커를 쓸 것. 영어 모델은 역효과",
        "verdict": "가장 확실한 개선. 먼저 시도할 기법",
    },
    {
        "method": "Ensemble",
        "best_for": "고유명사·약어·코드처럼 정확한 토큰 일치가 중요한 질의가 섞여 있을 때",
        "cost": "BM25 인덱스 추가 (거의 무료)",
        "caution": "약한 검색기를 섞으면 강한 쪽이 희석됨. 가중치를 실측으로 정할 것",
        "verdict": "평균보다 '실패 유형 보완'을 보고 판단",
    },
    {
        "method": "Parent Document",
        "best_for": "답변에 넓은 문맥이 필요한 긴 문서 (계약서, 논문, 매뉴얼)",
        "cost": "부모 청크 저장 공간",
        "caution": "Hit@k로는 효과가 안 보임. 답변 품질 지표로 평가할 것",
        "verdict": "검색 지표가 아니라 생성 품질로 판단할 기법",
    },
    {
        "method": "HyDE",
        "best_for": "LLM이 이미 잘 아는 일반 도메인 + 질문이 짧고 추상적일 때",
        "cost": "질의마다 LLM 호출 1회 (가장 비싸고 느림)",
        "caution": "전문/사내 용어 코퍼스에서는 LLM이 엉뚱한 분야를 지어내 역효과",
        "verdict": "도메인 적합성을 먼저 확인하고 도입",
    },
]

for g in guide:
    print(f"\n🔹 {g['method']}")
    print(f"   적합: {g['best_for']}")
    print(f"   비용: {g['cost']}")
    print(f"   주의: {g['caution']}")
    print(f"   판정: {g['verdict']}")

print(f"\n📌 실전 도입 순서")
print(f"  1. 임베딩 모델이 해당 언어를 지원하는지 확인 (여기서 대부분의 문제가 끝난다)")
print(f"  2. 정답 라벨 20~50개를 만들어 베이스라인을 측정")
print(f"  3. Reranking부터 적용 — 비용 대비 효과가 가장 크다")
print(f"  4. 실패한 질문을 유형별로 분류해 필요한 기법을 추가")
print(f"  5. 최종 판단은 답변 품질(RAGAS 등)로 (Session 08)")

In [ ]:
# 📌 실습 정리
print("📌 오늘의 핵심 정리")
print("=" * 60)
print("  1️⃣ 평가가 먼저다")
print("     · 검색기의 목적함수(코사인 유사도)를 평가에 쓰면")
print("       기본 RAG가 수학적으로 항상 이긴다 → 모든 기법이 나빠 보인다")
print("     · 정답 라벨 + Hit@k / MRR 로 평가해야 개선이 보인다")
print()
print("  2️⃣ 기법보다 모델이 먼저다")
print("     · 임베더: 영어 모델 MRR 0.175 → 한국어 모델 0.775 (4.4배)")
print("     · 리랭커: 같은 기법인데 영어 모델 Hit@1 45%, 다국어 모델 95%")
print()
print("  3️⃣ 기법별 실측 결과 (MRR@3 기준)")
print("     · Reranking      : +23%  크게 이김 — '순서' 문제를 해결")
print("     · Ensemble       :  +6%  단, 가중치를 0.5/0.5로 두면 -13%로 뒤집힘")
print("     · Parent Document:  +2%  순위는 본전, 컨텍스트는 1.5배")
print("     · HyDE           :  ~0%  실행마다 변동 = 개선 없음, 지연은 100배")
print()
print("  4️⃣ 설정 하나가 결론을 뒤집는다")
print("     · Ensemble 가중치가 같으면 RRF 동점이 생기고,")
print("       동점은 retrievers 리스트에 먼저 적은 쪽이 전부 가져간다")
print("     · 순서만 바꿔도 Hit@1이 60% ↔ 75% 로 달라진다")
print()
print("  5️⃣ 평균 하나로 판단하지 마라")
print("     · Ensemble의 이득은 고유명사 질문에 몰려 있다 (20% → 60%)")
print("     · Parent Document의 이득은 Hit@k에 아예 나타나지 않는다")
print()
print("  6️⃣ 비용을 함께 본다 — HyDE는 개선 없이 지연만 100배")
print()
print("  7️⃣ 최종 판단은 답변 품질로 → Session 08 (RAGAS)")
print("=" * 60)

---

## 🎯 실습 과제

1️⃣ **(평가의 함정 재현)** 평가 함수를 `mean cosine(query, retrieved)` 로 바꿔 다시 돌려보세요.
   모든 기법이 기본 RAG보다 나빠지는 것을 확인하고, 왜 그런지 설명해보세요.

2️⃣ **(모델이 기법을 좌우한다)** 리랭커를 `cross-encoder/ms-marco-MiniLM-L-6-v2`로 바꿔 측정하세요.
   같은 "리랭킹"인데 결과가 어떻게 달라지나요?

3️⃣ **(임베더 교체)** 임베딩 모델을 `all-MiniLM-L6-v2`로 되돌려 전체를 다시 측정하세요.
   기본 RAG와 Advanced 기법 중 어느 쪽이 더 크게 무너지나요?

4️⃣ **(앙상블 가중치와 동점)** Ensemble의 weight를 `[0.5, 0.5]`, `[0.3, 0.7]`, `[0.7, 0.3]`으로
   바꿔 비교하세요. `[0.5, 0.5]`일 때만 결과가 리스트 순서에 좌우되는 이유를 설명해보세요.
   또 `EnsembleRetriever(..., c=1)`로 바꾸면 순위 간 점수 격차가 어떻게 변하나요?

5️⃣ **(조합)** HyDE + Reranking을 조합해보세요. 리랭킹이 HyDE의 오염을 얼마나 복구해주나요?

6️⃣ **(직접 라벨링)** 자신의 문서 20개로 코퍼스를 만들고 질문 10개에 정답 출처를 라벨링해
   같은 벤치마크를 돌려보세요. 도메인이 바뀌면 승자도 바뀝니다.

---

## 📚 참고 자료
- [HyDE 논문 (Precise Zero-Shot Dense Retrieval without Relevance Labels)](https://arxiv.org/abs/2212.10496)
- [BGE Reranker (BAAI)](https://huggingface.co/BAAI/bge-reranker-v2-m3)
- [Sentence Transformers Cross-Encoders](https://www.sbert.net/docs/cross_encoder/usage/usage.html)
- [LangChain Ensemble Retriever](https://python.langchain.com/docs/how_to/ensemble_retriever/)
- [LangChain Parent Document Retriever](https://python.langchain.com/docs/how_to/parent_document_retriever/)
- [Reciprocal Rank Fusion 논문](https://plg.uwaterloo.ca/~gvcormac/cormacksigir09-rrf.pdf)
- [BEIR — 검색 벤치마크 데이터셋 모음](https://github.com/beir-cellar/beir)
